# Exploratory airport patterns (post hoc)

These analyses were done **after** the 2025 results were seen. They describe patterns that may help interpret the per-airport results (weather decreases at DFW/LAX, large gains at LGA/ORD, reference gains). Report them as exploratory: they test no pre-specified hypothesis and have no uncertainty intervals. Nothing is refitted; only saved predictions and processed data are read.

In [1]:
from pathlib import Path
import sys, importlib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import src.joint_bootstrap as jb
importlib.reload(jb)

RESULTS_DIR = PROJECT_ROOT / "outputs/final_evaluation"
OUT_DIR = RESULTS_DIR / "exploratory"
OUT_DIR.mkdir(exist_ok=True)
CORE = ["temperature", "dew_point_temperature", "wind_speed", "visibility"]
USE = ["Origin", "Dest", "DepDel15"] + CORE

manifest = pd.read_csv(RESULTS_DIR / "predictions/manifest.csv")
wide = jb.load_wide(PROJECT_ROOT, manifest)
flights = {year: pd.read_csv(PROJECT_ROOT / f"data/processed/flights_weather_{year}_top10.csv.gz", usecols=USE)
           for year in [2024, 2025]}
test = flights[2025]
for airport, frame in wide.items():   # prediction rows are indexed by row number in the 2025 file
    assert test.loc[frame.index, "Origin"].eq(airport).all()
    assert np.array_equal(test.loc[frame.index, "DepDel15"].to_numpy(), frame["DepDel15"].to_numpy())
print("Aligned predictions with 2025 features")

Aligned predictions with 2025 features


## Weather–delay profile per airport and year

Share of flights with adverse weather at the cutoff, delay rates with and without it, and the share of very cold or hot observations.

In [2]:
# Adverse = visibility < 5 km or wind speed >= 10 m/s at the prediction cutoff (GHCNh units).
def weather_profile(frame):
    adverse = frame["visibility"].lt(5) | frame["wind_speed"].ge(10)
    rate_adverse = frame.loc[adverse].groupby("Origin")["DepDel15"].mean()
    rate_other = frame.loc[~adverse].groupby("Origin")["DepDel15"].mean()
    return pd.DataFrame({
        "adverse_pct": 100 * adverse.groupby(frame["Origin"]).mean(),
        "delay_rate_adverse": rate_adverse,
        "delay_rate_other": rate_other,
        "delay_ratio": rate_adverse / rate_other,
        "below_zero_c_pct": 100 * frame["temperature"].le(0).groupby(frame["Origin"]).mean(),
        "at_least_35c_pct": 100 * frame["temperature"].ge(35).groupby(frame["Origin"]).mean(),
    })
profiles = pd.concat({year: weather_profile(frame) for year, frame in flights.items()}, names=["year"])
profiles.to_csv(OUT_DIR / "weather_delay_profiles.csv")
display(profiles.round(3))

adverse_pct  delay_rate_adverse  delay_rate_other  delay_ratio  \
year Origin                                                                   
2024 ATL           6.025               0.274             0.192        1.423   
     CLT           4.945               0.364             0.259        1.404   
     DEN           6.726               0.377             0.221        1.705   
     DFW           4.781               0.402             0.269        1.498   
     LAS           1.217               0.364             0.234        1.557   
     LAX           9.117               0.191             0.179        1.068   
     LGA           6.886               0.324             0.169        1.916   
     ORD           4.935               0.307             0.217        1.412   
     PHX           0.674               0.360             0.200        1.803   
     SEA           4.776               0.259             0.203        1.279   
2025 ATL           4.319               0.368             0.229        1.604   
     CLT           3.790               0.297             0.237        1.252   
     DEN           6.891               0.409             0.244        1.673   
     DFW           5.550               0.373             0.281        1.328   
     LAS           1.182               0.431             0.230        1.878   
     LAX           6.967               0.194             0.180        1.076   
     LGA           8.789               0.384             0.207        1.859   
     ORD           4.622               0.379             0.244        1.553   
     PHX           0.512               0.429             0.209        2.046   
     SEA           5.160               0.260             0.209        1.246   

             below_zero_c_pct  at_least_35c_pct  
year Origin                                      
2024 ATL                1.815             0.985  
     CLT                2.323             0.617  
     DEN               11.895             1.612  
     DFW                1.344             5.861  
     LAS                0.077            18.685  
     LAX                0.000             0.026  
     LGA                5.188             0.075  
     ORD               11.029             0.259  
     PHX                0.000            23.382  
     SEA                1.497             0.055  
2025 ATL                2.850             0.445  
     CLT                3.597             1.330  
     DEN               14.345             0.914  
     DFW                3.208             3.127  
     LAS                0.103            11.857  
     LAX                0.000             0.000  
     LGA               10.252             0.475  
     ORD               17.211             0.000  
     PHX                0.000            17.241  
     SEA                1.830             0.000

## Weather gain per month (unseen models)

AP(weather) − AP(flight-only) within each month. Monthly AP is noisy; use only to describe where gains or losses concentrate.

In [3]:
rows = []
for airport, frame in wide.items():
    month = frame["FlightDate"].dt.month
    for m in range(1, 13):
        subset = frame.loc[month.eq(m)]
        y = subset["DepDel15"].astype(int)
        row = {"airport": airport, "month": m, "flights": len(subset), "delay_rate": y.mean()}
        for family in ["logistic", "forest", "xgboost"]:
            row[family] = (average_precision_score(y, subset[f"{family}|weather|unseen"])
                           - average_precision_score(y, subset[f"{family}|flight_only|unseen"]))
        rows.append(row)
monthly = pd.DataFrame(rows)
monthly.to_csv(OUT_DIR / "monthly_weather_gain.csv", index=False)
display(monthly.loc[monthly["airport"].isin(["DFW", "LAX", "LGA", "ORD"])]
        .pivot(index="month", columns="airport", values=["forest", "xgboost", "logistic"]).round(3))

forest                      xgboost                      logistic  \
airport    DFW    LAX    LGA    ORD     DFW    LAX    LGA    ORD      DFW   
month                                                                       
1        0.035  0.010  0.041  0.033   0.066  0.009  0.055  0.036    0.039   
2        0.009  0.017  0.079  0.081   0.031  0.003  0.082  0.083    0.016   
3        0.008 -0.005  0.035  0.034   0.011 -0.002  0.052  0.037    0.027   
4       -0.032 -0.012 -0.005  0.007  -0.023 -0.007 -0.000  0.013    0.031   
5        0.023 -0.014  0.040  0.037   0.017 -0.018  0.065  0.042    0.040   
6       -0.025 -0.000  0.011  0.024  -0.031 -0.005  0.021  0.041   -0.005   
7       -0.012 -0.002  0.040  0.026  -0.010 -0.010  0.030  0.035   -0.002   
8       -0.023 -0.011  0.028  0.020  -0.013 -0.008  0.024  0.031    0.006   
9        0.005 -0.047  0.013  0.013   0.011 -0.053  0.008  0.005    0.015   
10      -0.016 -0.003  0.058 -0.007  -0.001 -0.013  0.079 -0.004    0.032   
11      -0.049 -0.014  0.043  0.126  -0.031 -0.012  0.036  0.131    0.048   
12      -0.018 -0.000  0.047  0.076  -0.024 -0.013  0.064  0.098    0.008   

                              
airport    LAX    LGA    ORD  
month                         
1        0.007  0.038  0.028  
2        0.020  0.066  0.032  
3       -0.000  0.039  0.037  
4       -0.004  0.002  0.017  
5       -0.002  0.065  0.022  
6        0.007  0.011 -0.004  
7       -0.000  0.006  0.007  
8        0.008  0.007  0.005  
9       -0.008  0.008  0.004  
10       0.018  0.073 -0.004  
11       0.002  0.044  0.055  
12       0.010  0.039  0.052

## Two candidate explanations checked

1. Missing weather: does the weather gain change when flights with missing weather are removed?
2. Unseen destinations: does the reference gain persist on flights whose destination occurs in the unseen training data?

In [4]:
rows = []
for airport, frame in wide.items():
    features = test.loc[frame.index]
    y = frame["DepDel15"].to_numpy().astype(int)
    missing = features[CORE].isna().any(axis=1).to_numpy()
    seen_dest = features["Dest"].isin(set(flights[2024].loc[flights[2024]["Origin"].ne(airport), "Dest"])).to_numpy()
    row = {"airport": airport, "missing_weather_pct": 100 * missing.mean(),
           "delay_rate_missing": y[missing].mean(), "unseen_dest_pct": 100 * (~seen_dest).mean()}
    for family in ["logistic", "forest", "xgboost"]:
        w, f = frame[f"{family}|weather|unseen"].to_numpy(), frame[f"{family}|flight_only|unseen"].to_numpy()
        row[f"weather_gain_{family}"] = average_precision_score(y, w) - average_precision_score(y, f)
        row[f"weather_gain_no_missing_{family}"] = (average_precision_score(y[~missing], w[~missing])
                                                   - average_precision_score(y[~missing], f[~missing]))
        r, u = frame[f"{family}|weather|reference"].to_numpy(), frame[f"{family}|weather|unseen"].to_numpy()
        row[f"reference_gain_{family}"] = average_precision_score(y, r) - average_precision_score(y, u)
        row[f"reference_gain_seen_dest_{family}"] = (average_precision_score(y[seen_dest], r[seen_dest])
                                                    - average_precision_score(y[seen_dest], u[seen_dest]))
    rows.append(row)
checks = pd.DataFrame(rows).set_index("airport")
checks.to_csv(OUT_DIR / "missing_weather_and_unseen_destinations.csv")
display(checks.round(4))

,missing_weather_pct,delay_rate_missing,unseen_dest_pct,weather_gain_logistic,weather_gain_no_missing_logistic,reference_gain_logistic,reference_gain_seen_dest_logistic,weather_gain_forest,weather_gain_no_missing_forest,reference_gain_forest,reference_gain_seen_dest_forest,weather_gain_xgboost,weather_gain_no_missing_xgboost,reference_gain_xgboost,reference_gain_seen_dest_xgboost
airport,,,,,,,,,,,,,,,
ATL,1.8826,0.1892,0.2978,0.0160,0.0158,0.0022,0.0022,0.0103,0.0101,0.0006,0.0006,0.0169,0.0165,0.0050,0.0050
CLT,1.4920,0.2128,0.1744,0.0079,0.0088,0.0013,0.0013,0.0145,0.0146,0.0028,0.0029,0.0169,0.0168,0.0079,0.0080
DEN,2.1371,0.2851,4.9400,0.0335,0.0339,0.0048,0.0050,0.0089,0.0094,0.0019,0.0018,0.0046,0.0044,0.0103,0.0103
DFW,1.6484,0.4144,6.4584,0.0118,0.0116,0.0081,0.0079,-0.0080,-0.0053,0.0186,0.0180,-0.0040,-0.0016,0.0230,0.0227
LAS,1.4685,0.2165,0.1727,0.0173,0.0194,0.0048,0.0049,0.0053,0.0050,0.0101,0.0101,0.0121,0.0118,0.0088,0.0088
LAX,1.7779,0.1653,0.0000,0.0072,0.0075,0.0027,0.0027,-0.0020,-0.0020,0.0083,0.0083,-0.0053,-0.0054,0.0125,0.0125
LGA,2.0000,0.2006,0.0577,0.0254,0.0243,0.0039,0.0039,0.0252,0.0250,0.0012,0.0012,0.0306,0.0301,0.0134,0.0134
ORD,1.5247,0.2002,3.7380,0.0142,0.0142,0.0038,0.0037,0.0290,0.0293,0.0007,0.0008,0.0306,0.0307,0.0084,0.0085
PHX,1.3617,0.2402,0.3271,0.0131,0.0134,0.0035,0.0035,0.0068,0.0074,0.0110,0.0110,0.0156,0.0157,0.0060,0.0060
